# OpenTelemetry × NeoMundi — quickstart

Measure a model call your application **already traces**, with one span
processor added next to your exporters — and see the one thing this
integration will not do: turn a span's token count into a measurement.

**This notebook costs nothing and opens no socket.** The provider is an
`httpx.MockTransport` behind the real `openai` client, instrumented by the
real contrib `opentelemetry-instrumentation-openai-v2`. The NeoMundi API is
a mock that replays a captured production response.

What it does **not** show: a real measurement persisted by the API. That
is outside this package's evidence (E1) — see `LIMITATIONS.md`.

## 1. Setup

In [1]:
import json
from importlib.metadata import version
from pathlib import Path

import httpx
import neomundi
from openai import OpenAI
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from otel_neomundi import NeoMundiSpanProcessor

for name in (
    "otel-neomundi",
    "opentelemetry-sdk",
    "opentelemetry-instrumentation-openai-v2",
    "neomundi",
):
    print(f"{name:42} {version(name)}")

otel-neomundi                              0.1.0
opentelemetry-sdk                          1.45.2
opentelemetry-instrumentation-openai-v2    2.4b0
neomundi                                   2.1.0


## 2. Your existing pipeline

A tracer provider with an exporter, and the openai-v2 instrumentation on
it. This is what an instrumented application already has; nothing here is
NeoMundi's.

In [2]:
provider = TracerProvider()
exporter = InMemorySpanExporter()  # stands in for your OTLP exporter
provider.add_span_processor(SimpleSpanProcessor(exporter))
OpenAIInstrumentor().instrument(tracer_provider=provider)


def model_client(report_usage: bool) -> OpenAI:
    def handle(request):
        body = {
            "id": "c",
            "object": "chat.completion",
            "created": 1,
            "model": "demo-model",
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": "Paris."},
                    "finish_reason": "stop",
                }
            ],
        }
        if report_usage:
            body["usage"] = {"prompt_tokens": 14, "completion_tokens": 3, "total_tokens": 17}
        return httpx.Response(200, json=body)

    return OpenAI(
        api_key="sk-demo",
        base_url="https://provider.example.invalid/v1",
        max_retries=0,
        http_client=httpx.Client(transport=httpx.MockTransport(handle)),
    )

## 3. Add the processor — next to the exporter, replacing nothing

`submit=False` first: no key, no socket, every check run locally.

In [3]:
processor = NeoMundiSpanProcessor(submit=False, tracer_provider=provider)
provider.add_span_processor(processor)

with processor.interaction() as interaction:
    reply = model_client(True).chat.completions.create(
        model="demo-model", messages=[{"role": "user", "content": "Capital of France?"}]
    )
    interaction.declare(token_count=reply.usage.total_tokens if reply.usage else None)

o = interaction.outcome
print(o.status, "| tokens", o.token_count, o.token_source, "| latency_ms", round(o.latency_ms, 3))

not_submitted_local_only | tokens 17 caller_declared | latency_ms 7.941


## 4. The refusal that defines this integration

The GenAI span below carries `gen_ai.usage.input_tokens = 14`. Nothing is
declared. **Nothing is measured.** A span attribute is written by an
instrumentation, not by the provider, and on 2026-11-26 we measured an
instrumentation option that replaces a provider's published count with a
tokenizer estimate while the span looks exactly the same.

In [4]:
with processor.interaction() as interaction:
    model_client(True).chat.completions.create(
        model="demo-model", messages=[{"role": "user", "content": "Capital of France?"}]
    )

genai = [s for s in exporter.get_finished_spans() if s.name.startswith("chat")][-1]
print("span usage :", {k: v for k, v in genai.attributes.items() if "usage" in k})
print("status     :", interaction.outcome.status)
print("reason     :", interaction.outcome.reason)

span usage : {'gen_ai.usage.input_tokens': 14, 'gen_ai.usage.output_tokens': 3}
status     : not_submitted_no_token_count
reason     : no token count was declared by the caller; the span carries gen_ai.usage.* but a span attribute cannot be traced to the provider, so it is never submitted -- declare the count the provider returned


## 5. Measured — and the request_id lands in your trace

With a client, the interaction is submitted, and `neomundi.request_id` is
set on the `neomundi.interaction` span, which your exporter receives in the
**same trace** as the model call.

In [5]:
FIXTURE = Path("..") / "tests" / "fixtures" / "rest_complete_allow.json"
PAYLOAD = {
    k: v
    for k, v in json.loads(FIXTURE.read_text(encoding="utf-8")).items()
    if not k.startswith("_fixture_")
}
SENT = []


def neomundi_api(request):
    SENT.append(json.loads(request.content))
    return httpx.Response(200, json=PAYLOAD)


client = neomundi.NeoMundiClient(
    api_key="ct_test_notebook_not_a_key",
    base_url="https://example.invalid",
    transport=httpx.MockTransport(neomundi_api),
)
measuring = NeoMundiSpanProcessor(client, tracer_provider=provider)
provider.add_span_processor(measuring)

with measuring.interaction() as interaction:
    reply = model_client(True).chat.completions.create(
        model="demo-model", messages=[{"role": "user", "content": "Capital of France?"}]
    )
    interaction.declare(token_count=reply.usage.total_tokens)

o = interaction.outcome
marker = [s for s in exporter.get_finished_spans() if s.name == "neomundi.interaction"][-1]
print("status            :", o.status)
print("sent token_count  :", SENT[-1]["raw_metrics"]["token_count"])
print("request_id        :", o.request_id)
print("in the trace      :", marker.attributes.get("neomundi.request_id"))
print("same trace        :", f"{marker.context.trace_id:032x}" == o.trace_id)

status            : submitted
sent token_count  : 17
request_id        : 00000000-0000-4000-8000-000000000331
in the trace      : 00000000-0000-4000-8000-000000000331
same trace        : True


## 6. What next

- Read `LIMITATIONS.md` — especially §1, why a span's count is refused.
- Point a client at the real API with `sample_rate` well below 1.0:
  **every submitted observation is billed**.

In [6]:
OpenAIInstrumentor().uninstrument()